# Generate Test Predictions for Competition Submission

This notebook generates predictions using **only the models that are actually trained**:
- Random Forest (3 encodings: onehot, blosum, aapc)
- Simple averaging of the 3 RF models

## Pipeline:
1. Load test data (59,540 sequences)
2. Extract features
3. Create 3 encodings (onehot, blosum, aapc)
4. Load trained RF models
5. Generate predictions from each model
6. Average predictions
7. Convert to binary (0/1)
8. Save submission.csv

## 1. Configuration

In [15]:
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file
TEST_INPUT = "../data/independent_test_input.csv"

# Output directory
OUTPUT_DIR = "../output/test_predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# RF Models (only what actually exists!)
#RF_MODELS = {
#    'onehot': "../output/rf/onehot/rf_onehot_models.pkl",
#    'blosum': "../output/rf/blosum/rf_blosum_models.pkl",
#    'aapc': "../output/rf/aapc/rf_aapc_models.pkl"
#}

# Prediction threshold
THRESHOLD = 0.5

print("Configuration loaded:")
print(f"  Test input: {TEST_INPUT}")
print(f"  Output: {OUTPUT_DIR}/submission.csv")
print(f"  Models: {len(RF_MODELS)} RF models")
print(f"  Threshold: {THRESHOLD}")

Configuration loaded:
  Test input: ../data/independent_test_input.csv
  Output: ../output/test_predictions/submission.csv
  Models: 3 RF models
  Threshold: 0.5


## 2. Import Libraries

In [16]:
import pandas as pd
import numpy as np
import pickle
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## 3. Load Test Data

In [17]:
# Load test data
test_df = pd.read_csv(TEST_INPUT)

print(f"✓ Loaded test data")
print(f"  Samples: {len(test_df):,}")
print(f"  Columns: {list(test_df.columns)}")
print(f"\nFirst few rows:")
test_df.head()

✓ Loaded test data
  Samples: 59,540
  Columns: ['ID', 'Sequence']

First few rows:


,ID,Sequence
0,Q0GA42,AAAAAAAALGVRLRDCCSRGAVLLLFFSLSP
1,Q0GA42,AAAAAAALGVRLRDCCSRGAVLLLFFSLSPR
2,P11047,AAAAAAGCAQAAMDECTDEGGRPQRCMPEFV
3,Q7TNS5,AAAAAASSASSPATRCKELGLAAAAAWEQQG
4,Q9NV92,AAAAAETSQRIQEEECPPRDDFSDADQLRVG


## 4. Feature Engineering

In [ ]:
# Physicochemical properties
HYDROPHOBICITY = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

MOLECULAR_WEIGHT = {
    'A': 89.09, 'R': 174.20, 'N': 132.12, 'D': 133.10, 'C': 121.15,
    'Q': 146.15, 'E': 147.13, 'G': 75.07, 'H': 155.16, 'I': 131.17,
    'L': 131.17, 'K': 146.19, 'M': 149.21, 'F': 165.19, 'P': 115.13,
    'S': 105.09, 'T': 119.12, 'W': 204.23, 'Y': 181.19, 'V': 117.15
}

CHARGE = {
    'A': 0, 'R': 1, 'N': 0, 'D': -1, 'C': 0,
    'Q': 0, 'E': -1, 'G': 0, 'H': 0.1, 'I': 0,
    'L': 0, 'K': 1, 'M': 0, 'F': 0, 'P': 0,
    'S': 0, 'T': 0, 'W': 0, 'Y': 0, 'V': 0
}

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

def extract_features(sequence):
    """Extract all features from a sequence."""
    features = {}
    counter = Counter(sequence)
    total = len(sequence)
    
    # 1. Amino acid composition (20 features)
    for aa in AMINO_ACIDS:
        features[f'aa_{aa}'] = counter.get(aa, 0) / total
    
    # 2. Cysteine features
    cys_positions = [i for i, aa in enumerate(sequence) if aa == 'C']
    features['cys_count'] = len(cys_positions)
    features['cys_present'] = 1 if len(cys_positions) > 0 else 0
    
    if len(cys_positions) > 0:
        features['cys_first_pos'] = cys_positions[0]
        middle_dist = min([abs(pos - 15) for pos in cys_positions])
        features['cys_middle_dist'] = middle_dist
        
        # Context around first cysteine
        first_cys = cys_positions[0]
        for offset in [-2, -1, 1, 2]:
            pos = first_cys + offset
            if 0 <= pos < len(sequence):
                features[f'cys_ctx_{offset}'] = sequence[pos]
            else:
                features[f'cys_ctx_{offset}'] = 'X'
    else:
        features['cys_first_pos'] = -1
        features['cys_middle_dist'] = 31
        for offset in [-2, -1, 1, 2]:
            features[f'cys_ctx_{offset}'] = 'X'
    
    # 3. Physicochemical properties
    hydro_scores = [HYDROPHOBICITY.get(aa, 0) for aa in sequence]
    features['hydro_mean'] = np.mean(hydro_scores)
    features['hydro_std'] = np.std(hydro_scores)
    
    mw_scores = [MOLECULAR_WEIGHT.get(aa, 0) for aa in sequence]
    features['mw_mean'] = np.mean(mw_scores)
    
    charge_scores = [CHARGE.get(aa, 0) for aa in sequence]
    features['charge_total'] = np.sum(charge_scores)
    
    # 4. Position features
    features['middle_aa'] = sequence[15]
    
    # 5. Complexity
    features['unique_aa'] = len(set(sequence))
    
    return features

print("✓ Feature extraction function defined")

In [ ]:
print(f"Extracting features from {len(test_df):,} sequences...\n")

features_list = []
for idx, row in test_df.iterrows():
    if (idx + 1) % 10000 == 0:
        print(f"  Processed {idx+1:,} / {len(test_df):,} sequences...")
    features_list.append(extract_features(row['Sequence']))

features_df = pd.DataFrame(features_list)

print(f"\n✓ Feature extraction complete!")
print(f"  Features: {len(features_df.columns)}")
print(f"  Shape: {features_df.shape}")

## 5. Create Encoded Feature Sets

In [ ]:
def blosum_encode(aa):
    """Encode amino acid as integer."""
    mapping = {aa: i for i, aa in enumerate('ARNDCQEGHILKMFPSTWYV')}
    mapping['X'] = 20
    return mapping.get(aa, 20)

# Separate categorical and numerical features
categorical_cols = ['middle_aa', 'cys_ctx_-2', 'cys_ctx_-1', 'cys_ctx_1', 'cys_ctx_2']
numerical_cols = [c for c in features_df.columns if c not in categorical_cols]

print("Creating encoded feature sets...\n")

# 1. ONE-HOT encoding
features_onehot = features_df.copy()
for col in categorical_cols:
    dummies = pd.get_dummies(features_onehot[col], prefix=col, drop_first=False)
    features_onehot = pd.concat([features_onehot.drop(columns=[col]), dummies], axis=1)
print(f"✓ One-hot encoding: {features_onehot.shape[1]} features")

# 2. BLOSUM encoding
features_blosum = features_df[numerical_cols].copy()
for col in categorical_cols:
    features_blosum[col] = features_df[col].apply(blosum_encode)
print(f"✓ BLOSUM encoding: {features_blosum.shape[1]} features")

# 3. AAPC (numerical only)
features_aapc = features_df[numerical_cols].copy()
print(f"✓ AAPC encoding: {features_aapc.shape[1]} features")

# Store in dictionary
encoded_features = {
    'onehot': features_onehot,
    'blosum': features_blosum,
    'aapc': features_aapc
}

print(f"\n✓ All encodings prepared!")

## 6. Load RF Models and Generate Predictions

In [ ]:
print("="*80)
print("LOADING MODELS AND GENERATING PREDICTIONS")
print("="*80)

label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]
predictions = {}

for encoding, model_file in RF_MODELS.items():
    print(f"\n[{encoding.upper()}]")
    
    if not os.path.exists(model_file):
        print(f"  ✗ Model file not found: {model_file}")
        continue
    
    # Load models (RF saves dict directly)
    with open(model_file, 'rb') as f:
        models = pickle.load(f)
    print(f"  ✓ Loaded models")
    
    # Get features for this encoding
    X_test = encoded_features[encoding]
    print(f"  Features shape: {X_test.shape}")
    
    # Generate predictions for each label
    pred_proba = np.zeros((len(X_test), 3))
    for i, label in enumerate(label_cols):
        pred_proba[:, i] = models[label].predict_proba(X_test)[:, 1]
    
    predictions[f"rf_{encoding}"] = pred_proba
    print(f"  ✓ Predictions generated: {pred_proba.shape}")
    print(f"  Probability range: [{pred_proba.min():.3f}, {pred_proba.max():.3f}]")

print(f"\n{'='*80}")
print(f"✓ Total models loaded: {len(predictions)}")
print(f"{'='*80}")

## 7. Ensemble: Simple Average

In [ ]:
print("\nAveraging predictions from all models...")

# Simple average of all RF models
ensemble_pred = np.mean(list(predictions.values()), axis=0)

print(f"✓ Ensemble predictions generated")
print(f"  Shape: {ensemble_pred.shape}")
print(f"  Probability range: [{ensemble_pred.min():.4f}, {ensemble_pred.max():.4f}]")

## 8. Convert to Binary Predictions

In [ ]:
print(f"\nApplying threshold = {THRESHOLD}...")

# Convert probabilities to binary (0 or 1)
binary_pred = (ensemble_pred > THRESHOLD).astype(int)

print(f"✓ Binary predictions generated")
print(f"  Shape: {binary_pred.shape}")
print(f"\nPredicted positive rates:")
for i, label in enumerate(label_cols):
    pos_count = binary_pred[:, i].sum()
    pos_rate = pos_count / len(binary_pred) * 100
    print(f"  {label}: {pos_count:,} ({pos_rate:.2f}%)")

## 9. Create Submission File

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'S-glutathionylation': binary_pred[:, 0],
    'S-nitrosylation': binary_pred[:, 1],
    'S-palmitoylation': binary_pred[:, 2]
})

# Save to CSV
submission_file = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(submission_file, index=False)

print("\n" + "="*80)
print("✓ SUBMISSION FILE CREATED!")
print("="*80)
print(f"\nFile: {submission_file}")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
display(submission.head(10))

print(f"\n{'='*80}")
print("✓ Ready for submission!")
print(f"{'='*80}")

## 10. Optional: Save Probabilities

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Sequence': test_df['Sequence'],  # Include Sequence column
    'S-glutathionylation': binary_pred[:, 0],
    'S-nitrosylation': binary_pred[:, 1],
    'S-palmitoylation': binary_pred[:, 2]
})

# Save to CSV
submission_file = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(submission_file, index=False)

print("\n" + "="*80)
print("✓ SUBMISSION FILE CREATED!")
print("="*80)
print(f"\nFile: {submission_file}")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
display(submission.head(10))

print(f"\n{'='*80}")
print("✓ Ready for submission!")
print(f"{'='*80}")